# 00 Prepare the data

The other notebooks in this folder read from `cache/`,
so they all use the same data and the same test set.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

DATA_FILE = Path("../data/user_top_tracks.csv")
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

N_USERS = None                # None = all users, e.g. 20_000 for a quick test
N_HIDDEN = 10                 # tracks hidden per user
MIN_USERS_PER_TRACK = 20      # rarer tracks are not used
EMBEDDING_DIM = 8             # size of the user vectors used for clustering
N_EVAL_USERS = 10_000         # users that get recommendations
SEED = 579

## Load and clean

In [2]:
start = time.perf_counter()
df = pd.read_csv(DATA_FILE,
                 usecols=["user_id", "track_name", "artist_name", "playcount"],
                 nrows=N_USERS * 50 if N_USERS else None,
                 dtype={"user_id": "int32", "playcount": "int32", "track_name": "category", "artist_name": "category"},
                 keep_default_na=False, na_values=[""])
n_rows = len(df)
df = df.dropna(subset=["track_name", "artist_name"])
n_empty = n_rows - len(df)
df = df[df["artist_name"] != "[unknown]"]
n_unknown = n_rows - n_empty - len(df)

# number users and tracks 0, 1, 2, ...
df["user"] = df["user_id"].astype("category").cat.codes.astype("int32")
df["track"] = df.groupby(["track_name", "artist_name"], observed=True).ngroup().astype("int32")
n_users = df["user"].max() + 1
n_tracks = df["track"].max() + 1

print(f"{n_rows:,} rows in the file: {n_empty:,} removed (empty title or artist), {n_unknown:,} removed ([unknown] artist)")
print(f"{n_users:,} users, {len(df):,} rows, {n_tracks:,} tracks ({time.perf_counter() - start:.0f}s)")

23,822,550 rows in the file: 0 removed (empty title or artist), 7,321 removed ([unknown] artist)
476,451 users, 23,815,229 rows, 3,325,347 tracks (70s)


## Hide 10 tracks per user

In [ ]:
start = time.perf_counter()
df = df.sample(frac=1, random_state=SEED)
position = df.groupby("user").cumcount()
n_tracks_of_user = df.groupby("user")["track"].transform("size")
df["hidden"] = (position < N_HIDDEN) & (n_tracks_of_user >= MIN_USERS_PER_TRACK)

print(f"hidden rows: {df['hidden'].sum():,} ({time.perf_counter() - start:.0f}s)")

hidden rows: 4,764,400 (4s)


## User x track matrix
Only training rows.

In [4]:
train = df[~df["hidden"]]
users_per_track = np.bincount(train["track"], minlength=n_tracks)
vocabulary = np.flatnonzero(users_per_track >= MIN_USERS_PER_TRACK)      # track ids that become columns

column_of_track = np.full(n_tracks, -1)                                 # track id -> column (-1 = not used)
column_of_track[vocabulary] = np.arange(len(vocabulary))

n_train = len(train)
train = train[column_of_track[train["track"]] >= 0]
print(f"{n_train - len(train):,} of {n_train:,} training rows removed (track has fewer than {MIN_USERS_PER_TRACK} users)")
ratings = sp.csr_matrix(
    (np.log1p(train["playcount"].to_numpy()).astype(np.float32),
     (train["user"].to_numpy(), column_of_track[train["track"]])),
    shape=(n_users, len(vocabulary)))

print(f"matrix: {ratings.shape[0]:,} users x {ratings.shape[1]:,} tracks, {ratings.nnz:,} non-zeros")
print(f"users with no track in the matrix: {(ratings.getnnz(axis=1) == 0).sum():,}")

6,004,246 of 19,050,829 training rows removed (track has fewer than 20 users)


matrix: 476,451 users x 120,650 tracks, 13,046,578 non-zeros
users with no track in the matrix: 2,948


## Embedding for clustering

In [5]:
start = time.perf_counter()
svd = TruncatedSVD(n_components=EMBEDDING_DIM, random_state=SEED)
embedding = normalize(svd.fit_transform(normalize(ratings)))
print(f"embedding: {embedding.shape}, explained variance {svd.explained_variance_ratio_.sum():.1%} "
      f"({time.perf_counter() - start:.1f}s)")

embedding: (476451, 8), explained variance 1.2% (1.6s)


## Users to evaluate

In [6]:
rng = np.random.default_rng(SEED)
users_with_hidden = np.unique(df.loc[df["hidden"], "user"])
eval_users = np.sort(rng.choice(users_with_hidden, size=min(N_EVAL_USERS, len(users_with_hidden)), replace=False))

hidden = df[df["hidden"] & df["user"].isin(eval_users)].sort_values("user")
hidden_items = column_of_track[hidden["track"]].reshape(len(eval_users), N_HIDDEN)

print(f"{len(eval_users):,} evaluated users")
print(f"hidden tracks that can be recommended: {(hidden_items >= 0).mean():.1%}")

10,000 evaluated users
hidden tracks that can be recommended: 67.7%


## Save

In [7]:
sp.save_npz(CACHE_DIR / "ratings.npz", ratings)
np.save(CACHE_DIR / "embedding.npy", embedding)
np.save(CACHE_DIR / "eval_users.npy", eval_users)
np.save(CACHE_DIR / "hidden_items.npy", hidden_items)

first_row = df[df["track"].isin(vocabulary)].drop_duplicates("track").set_index("track").loc[vocabulary]
track_names = first_row["track_name"].astype(str) + " — " + first_row["artist_name"].astype(str)
pd.Series(track_names.to_numpy(), name="track").to_csv(CACHE_DIR / "tracks.csv", index=False)
print("saved to", CACHE_DIR)

saved to cache
